# Results and significance

Aggregates the per-run results into the paper's Results and Discussion sections: mean test
MSE and mean total tuning wall-clock (+/- SD across the ten seeds) per (dataset, optimizer),
ranked within each dataset, a Friedman test across the 20 dataset-blocks, and
all-pairs Wilcoxon signed-rank tests with a Holm step-down correction where the omnibus
rejects. This is the procedure of Demsar (2006) that the paper's Statistical Procedure
section describes, and it is implemented once in `methodology.rank_and_friedman`.

Aggregation is the mean across seeds, with the SD it is computed from shown alongside.

In [1]:
import pandas as pd

import methodology as meth

ALGORITHMS = meth.ALGORITHMS
GRADIENT_OPTIMIZERS = meth.GRADIENT_ALGORITHMS
ALPHA = 0.05

panel = meth.load_panel()
print(f"Panel: {panel['dataset'].nunique()} datasets, {len(panel)} rows, "
      f"N = {int(panel['n_samples'].min())}-{int(panel['n_samples'].max())}")
panel.tail()

Panel: 20 datasets, 9200 rows, N = 178-6497


,Unnamed: 0,dataset,optimizer,seed,init,cfg,val_mse,test_mse,nfev,njev,...,nit,wall_clock,stop_reason,t_max,kappa,eta,kappa_dim,expected_kappa_dim,search_budget,n_samples
9195,9195,UCI: Wine Quality,L-BFGS-B,9,1.0,{'maxcor': 7},0.566417,NaN,8,8,...,2.0,1.925243,time_budget,22.950034,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.75056526...",11.563020,11,11,256.0,6497.0
9196,9196,UCI: Wine Quality,L-BFGS-B,9,2.0,{'maxcor': 7},0.396865,NaN,8,8,...,2.0,1.924021,time_budget,22.950034,"[20.41125620191451, 21.315158397017512, 20.543...",30.278695,11,11,256.0,6497.0
9197,9197,UCI: Wine Quality,L-BFGS-B,9,3.0,{'maxcor': 7},0.389592,NaN,9,9,...,4.0,2.165744,time_budget,22.950034,"[46.30238767959057, 50.36357938575389, 28.3130...",25.766039,11,11,256.0,6497.0
9198,9198,UCI: Wine Quality,L-BFGS-B,9,4.0,{'maxcor': 7},0.400460,NaN,11,11,...,2.0,2.645587,time_budget,22.950034,"[60.81910781618549, 59.50959419069791, 59.6698...",27.982132,11,11,256.0,6497.0
9199,9199,UCI: Wine Quality,L-BFGS-B,9,5.0,{'maxcor': 7},0.398711,NaN,13,13,...,2.0,3.129130,time_budget,22.950034,"[78.57731670011248, 78.46059084993797, 80.0, 7...",21.168435,11,11,256.0,6497.0


## Accuracy: aggregate, rank, Friedman

Mean test MSE (+/- SD across the ten seeds) per (dataset, optimizer).

In [2]:
acc = meth.per_group_accuracy(panel)
mean_mse = meth.wide_mean(acc, "test_mse")
std_mse = meth.wide_sd(acc, "test_mse")
winner = mean_mse[ALGORITHMS].idxmin(axis=1)

print("Mean test MSE and SD across the ten seeds, with the per-dataset winner (lowest mean), 4 s.f.:")
display_mse = pd.concat(
    {alg: pd.DataFrame({"mean": mean_mse[alg], "sd": std_mse[alg]}) for alg in ALGORITHMS},
    axis=1,
).map(meth.fmt_sigfig)
display_mse["winner"] = winner
display_mse

Mean test MSE and SD across the ten seeds, with the per-dataset winner (lowest mean), 4 s.f.:


Adam             L-BFGS           L-BFGS-B  \
                               mean        sd     mean        sd     mean   
dataset                                                                     
Concrete                      49.38     7.280    50.36     8.317    49.90   
Forest Fires                   7552      6201     6662      5249     7768   
Friedman1                     110.7     14.21    110.0     16.23    112.6   
Friedman2                     589.4     78.20    596.8     72.78    594.1   
Moons                       0.02065  0.009806  0.02069  0.009226  0.01483   
Polynomial Regression       0.04286  0.007792  0.04306  0.008573  0.04233   
Regression                   0.5763    0.8537   0.5765    0.8451   0.5762   
UCI: Airfoil Self-Noise       10.88     1.448    11.99     1.448    10.52   
UCI: Auto MPG                 9.523     4.429    10.67     5.418    10.15   
UCI: Blood Transfusion       0.1664   0.01263   0.1633   0.01149   0.1632   
UCI: Energy Efficiency       0.2434   0.03681   0.3467   0.08272   0.2330   
UCI: Glass Identification     1.245    0.5971    1.153    0.4964    1.643   
UCI: Heart Failure           0.1230   0.03626   0.1220   0.03566   0.1375   
UCI: Ionosphere             0.08066   0.02741  0.07292   0.01608  0.07009   
UCI: Parkinsons             0.06574   0.03884  0.08340   0.05137  0.05511   
UCI: Real Estate Valuation    67.05     31.52    60.32     32.18    76.87   
UCI: Sonar                   0.1501   0.04531   0.1450   0.05576   0.1602   
UCI: Statlog Heart           0.1661   0.03866   0.1630   0.04552   0.1993   
UCI: Wine                   0.09315   0.04114  0.08827   0.03935  0.07480   
UCI: Wine Quality            0.4363   0.01671   0.4349   0.02263   0.4349   

                                     Grid search                 winner  
                                  sd        mean        sd               
dataset                                                                  
Concrete                       7.902       57.82     6.770         Adam  
Forest Fires                    6126        6886      5740       L-BFGS  
Friedman1                      14.14       108.9     16.53  Grid search  
Friedman2                      69.93       855.8     112.8         Adam  
Moons                       0.008735     0.01966  0.008961     L-BFGS-B  
Polynomial Regression       0.007786     0.05710   0.01068     L-BFGS-B  
Regression                    0.8452      0.6598    0.9924     L-BFGS-B  
UCI: Airfoil Self-Noise        1.412       13.84     1.974     L-BFGS-B  
UCI: Auto MPG                  5.522       9.118     3.932  Grid search  
UCI: Blood Transfusion       0.01168      0.1638   0.01144     L-BFGS-B  
UCI: Energy Efficiency       0.03330      0.3514   0.08709     L-BFGS-B  
UCI: Glass Identification     0.6304       1.065    0.6051  Grid search  
UCI: Heart Failure           0.04553      0.1210   0.03533  Grid search  
UCI: Ionosphere              0.02155     0.07066   0.01713     L-BFGS-B  
UCI: Parkinsons              0.04207     0.07161   0.02300     L-BFGS-B  
UCI: Real Estate Valuation     33.65       61.17     34.70       L-BFGS  
UCI: Sonar                   0.05572      0.1172   0.02247  Grid search  
UCI: Statlog Heart           0.04442      0.1395   0.03126  Grid search  
UCI: Wine                    0.04830     0.07553   0.03715     L-BFGS-B  
UCI: Wine Quality            0.02251      0.4326   0.01777  Grid search

### Win counts

How many of the 20 datasets each method wins on mean test MSE. A tally of the `winner` column
above

In [3]:
win_counts = winner.value_counts().reindex(ALGORITHMS, fill_value=0)
assert win_counts.sum() == mean_mse.shape[0] == 20

print(f"Win counts across all {mean_mse.shape[0]} datasets (by mean test MSE):")
win_counts.to_frame(name="wins").T

Win counts across all 20 datasets (by mean test MSE):


,Adam,L-BFGS,L-BFGS-B,Grid search
wins,2,2,9,7


In [4]:
mse_stats = meth.rank_and_friedman(mean_mse, ALGORITHMS, alpha=ALPHA)
print(f"Friedman chi-square = {meth.fmt_sigfig(mse_stats['friedman_stat'])}, "
      f"p = {meth.fmt_sigfig(mse_stats['p_friedman'])}")
print(f"Average rank across {mean_mse.shape[0]} datasets (lower = better):")
print(mse_stats["avg_rank"].apply(meth.fmt_sigfig).to_string())

if mse_stats["significant"]:
    print(f"\np < {ALPHA}: the omnibus test rejects; see the all-pairs post-hoc below.")
else:
    print(f"\np >= {ALPHA}: fail to reject H0. This is a failure to detect a difference in "
          f"accuracy, not a demonstration that the methods are equivalent.")

mse_stats["posthoc"]

Friedman chi-square = 1.140, p = 0.7674
Average rank across 20 datasets (lower = better):
optimizer
L-BFGS-B       2.350
Grid search    2.350
L-BFGS         2.600
Adam           2.700

p >= 0.05: fail to reject H0. This is a failure to detect a difference in accuracy, not a demonstration that the methods are equivalent.


## Cost: aggregate, rank, Friedman

The same pipeline applied to total tuning wall-clock per group

In [5]:
stop_reason_counts = (
    panel[panel["optimizer"].isin(GRADIENT_OPTIMIZERS)]
    .groupby(["optimizer", "stop_reason"]).size().unstack(fill_value=0)
)
print("stop_reason per gradient method, across all (dataset, seed, cfg, init) runs:")
stop_reason_counts

stop_reason per gradient method, across all (dataset, seed, cfg, init) runs:


stop_reason,abnormal,converged,time_budget
optimizer,,,
Adam,0,158,2842
L-BFGS,0,1976,1024
L-BFGS-B,10,2151,839


In [6]:
cost = meth.per_group_cost(panel)
mean_wall_clock = meth.wide_mean(cost, "wall_clock")
std_wall_clock = meth.wide_sd(cost, "wall_clock")
fastest = mean_wall_clock[ALGORITHMS].idxmin(axis=1)

print("Mean total tuning wall-clock and SD across the ten seeds (seconds), 4 s.f.:")
display_wc = pd.concat(
    {alg: pd.DataFrame({"mean": mean_wall_clock[alg], "sd": std_wall_clock[alg]}) for alg in ALGORITHMS},
    axis=1,
).map(meth.fmt_sigfig)
display_wc["fastest"] = fastest
display_wc

Mean total tuning wall-clock and SD across the ten seeds (seconds), 4 s.f.:


Adam           L-BFGS          L-BFGS-B  \
                              mean       sd    mean       sd     mean   
dataset                                                                 
Concrete                     3.807  0.09104   2.975   0.1293    3.571   
Forest Fires                 4.086   0.3602   2.691   0.4669    3.635   
Friedman1                    2.615  0.06770   1.957   0.1667    2.152   
Friedman2                    2.194  0.03827   1.757  0.08144    1.812   
Moons                        1.485  0.08214  0.1725  0.05941   0.2267   
Polynomial Regression       0.9219  0.03556  0.5099  0.06949   0.2716   
Regression                  0.8963  0.05397  0.1299  0.07964   0.1514   
UCI: Airfoil Self-Noise      5.177  0.06562   3.972   0.2611    4.868   
UCI: Auto MPG                1.241  0.02527  0.9511  0.09224   0.9941   
UCI: Blood Transfusion       1.371  0.07477  0.1119  0.02761  0.06690   
UCI: Energy Efficiency       2.438   0.1077  0.5268   0.1809    1.040   
UCI: Glass Identification    1.257  0.05952  0.9385  0.08126    1.130   
UCI: Heart Failure           1.534  0.05009  0.4783   0.1256   0.7758   
UCI: Ionosphere              3.917  0.05492  0.5408   0.1932    1.055   
UCI: Parkinsons              2.601  0.07827  0.6169   0.2091   0.7037   
UCI: Real Estate Valuation   1.105  0.03299  0.8314  0.08906   0.8579   
UCI: Sonar                   5.631  0.07572   2.207   0.6687    3.006   
UCI: Statlog Heart           1.682  0.07270  0.6146   0.1885   0.8465   
UCI: Wine                    1.547   0.1179  0.3796   0.1933   0.3828   
UCI: Wine Quality            25.20  0.05779   18.25   0.9045    34.31   

                                    Grid search             fastest  
                                 sd        mean        sd            
dataset                                                              
Concrete                     0.1663       3.775    0.1026    L-BFGS  
Forest Fires                 0.4622       4.230    0.5969    L-BFGS  
Friedman1                    0.1714       2.605   0.05833    L-BFGS  
Friedman2                    0.1694       2.192   0.01116    L-BFGS  
Moons                       0.04812       1.561   0.01414    L-BFGS  
Polynomial Regression       0.04355      0.9435  0.002196  L-BFGS-B  
Regression                  0.06786      0.9439  0.002580    L-BFGS  
UCI: Airfoil Self-Noise      0.2599       5.131   0.01928    L-BFGS  
UCI: Auto MPG               0.08039       1.237   0.02515    L-BFGS  
UCI: Blood Transfusion      0.03540       1.418   0.04017  L-BFGS-B  
UCI: Energy Efficiency       0.1853       2.418    0.1052    L-BFGS  
UCI: Glass Identification    0.1062       1.244   0.05920    L-BFGS  
UCI: Heart Failure           0.1962       1.536   0.05780    L-BFGS  
UCI: Ionosphere              0.4984       3.894   0.04698    L-BFGS  
UCI: Parkinsons              0.3357       2.618   0.02105    L-BFGS  
UCI: Real Estate Valuation  0.08820       1.097   0.03118    L-BFGS  
UCI: Sonar                    1.060       5.582   0.08236    L-BFGS  
UCI: Statlog Heart           0.2044       1.691   0.05940    L-BFGS  
UCI: Wine                    0.2179       1.609  0.006789    L-BFGS  
UCI: Wine Quality             1.978       22.82   0.05958    L-BFGS

In [7]:
wc_stats = meth.rank_and_friedman(mean_wall_clock, ALGORITHMS, alpha=ALPHA)
print(f"Friedman chi-square = {meth.fmt_sigfig(wc_stats['friedman_stat'])}, "
      f"p = {meth.fmt_sigfig(wc_stats['p_friedman'])}")
print(f"Average speed rank across {mean_wall_clock.shape[0]} datasets (lower = faster):")
print(wc_stats["avg_rank"].apply(meth.fmt_sigfig).to_string())
print(f"\nFastest on {fastest.value_counts().to_dict()} of the 20 datasets.")

wc_posthoc = wc_stats["posthoc"]
if wc_posthoc is not None:
    wc_posthoc = wc_posthoc.copy()
    for col in ("W", "p_raw", "p_holm"):
        wc_posthoc[col] = wc_posthoc[col].apply(meth.fmt_sigfig)
wc_posthoc

Friedman chi-square = 48.24, p = 1.893e-10
Average speed rank across 20 datasets (lower = faster):
optimizer
L-BFGS         1.100
L-BFGS-B       2.000
Grid search    3.400
Adam           3.500

Fastest on {'L-BFGS': 18, 'L-BFGS-B': 2} of the 20 datasets.


,comparison,W,p_raw,p_holm,significant
0,Adam vs L-BFGS,0.000,1.907e-06,1.144e-05,True
1,Adam vs L-BFGS-B,20.00,0.0007076,0.002123,True
2,Adam vs Grid search,103.0,0.9563,0.9563,False
3,L-BFGS vs L-BFGS-B,17.00,0.0003948,0.001579,True
4,L-BFGS vs Grid search,0.000,1.907e-06,1.144e-05,True
5,L-BFGS-B vs Grid search,20.00,0.0007076,0.002123,True


## Adam's beta1, beta2 and epsilon

`beta1 = 0.9`, `beta2 = 0.999` and `epsilon = 1e-8` are Kingma and Ba's published defaults,
left unchanged so that every gradient method is given the same number of tuned axes. The
paper's Limitations section argues this is low-risk here: those moments exist to smooth
stochastic minibatch gradient noise, and these gradients are exact and full-batch, so there is
no sampling noise for them to average away. Their remaining role is the per-coordinate
rescaling `m_hat / (sqrt(v_hat) + epsilon)`, which is the diagonal-only curvature adaptation
that the Discussion contrasts against L-BFGS's off-diagonal approximation.

The cell below records the one place where the defaults sit awkwardly against this protocol:
`beta2 = 0.999` implies a second-moment averaging horizon of `1 / (1 - beta2)` iterations,
which is far longer than the number of iterations Adam actually completes inside its share of
`t_max`.

In [8]:
ADAM_BETA2 = 0.999  # kappaeta.AdamOptimizer default
beta2_horizon = 1 / (1 - ADAM_BETA2)

adam_nit = panel.loc[panel["optimizer"] == "Adam", "nit"]
print(f"Adam iterations completed across all {len(adam_nit)} (dataset, seed, cfg, init) runs:")
print(adam_nit.describe().apply(meth.fmt_sigfig).to_string())

share_under_horizon = (adam_nit < beta2_horizon).mean()
print(f"\nbeta2 = {ADAM_BETA2} implies an averaging horizon of 1/(1-beta2) = {beta2_horizon:.0f} iterations.")
print(f"Share of runs completing fewer iterations than that horizon: {share_under_horizon:.2%}")

Adam iterations completed across all 3000 (dataset, seed, cfg, init) runs:
count     3000
mean     54.16
std      12.79
min      2.000
25%      53.00
50%      57.00
75%      60.00
max      87.00

beta2 = 0.999 implies an averaging horizon of 1/(1-beta2) = 1000 iterations.
Share of runs completing fewer iterations than that horizon: 100.00%


Every Adam run finishes in far fewer iterations than `beta2`'s averaging horizon, so the
second-moment estimate never leaves its bias-corrected warm-up transient. `beta2 = 0.999` is
calibrated for training regimes an order of magnitude longer than this protocol runs. That is
a candidate follow-up rather than a correction to make here: retuning it would only change a
conclusion if Adam's standing against the quasi-Newton methods were decision-critical, and on
accuracy no method is separated from any other.

## What the two tests together support

Every method pays the same dominant cost per evaluation, `O(MN^2 d)` for the loss or gradient,
and the overhead each adds on top of it is small: `O(d)` for Adam's moment updates and
`O(m(d+1))` for L-BFGS's two-loop recursion. The entire difference between them is therefore
how many evaluations they need, not what each evaluation costs.

The mechanism is the one the quasi-Newton literature predicts for anisotropic surfaces. In
this loss, `eta` controls prediction smoothness and typically has high curvature while
individual `kappa_p` components may have much lower curvature, and the optimal step in one
depends on the current value of the other. Adam scales each axis independently and has no
representation for that coupling. L-BFGS's implicit inverse-Hessian scaling rotates and scales
the gradient to the local geometry, and its secant approximation captures the off-diagonal
interactions a diagonal method discards -- at a per-iteration cost that is negligible beside
the gradient evaluation both methods pay anyway.

The two tests above locate where that advantage appears. Under a matched budget of five
initialisations and three configurations, all four searches have enough evaluations to reach
the same basin on most of these datasets, and no accuracy difference is detectable. On cost
the difference is decisive. What the quasi-Newton methods buy is not a better solution but the
same solution sooner, which is why the paper recommends the class as the default for
kappa-eta tuning: L-BFGS as the cheapest member, and L-BFGS-B as the safer one when the
optimum may lie on the parameter bounds, since it solves the constrained problem rather than
clipping after the step.